# Trabajo Práctico Final — Introducción a la Ingeniería de Datos

**Autora:** Bioing. Aylin Agatha Vazquez Chenlo — Doctorado en Ingeniería, ITBA

**Profesor:** Dr. Nicolás Guillermo Tripp

**Repositorio:** https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos

---

## Declaración de uso de IA

- "Claude Opus 5.5 (Anthropic) para buscar, comparar y evaluar datasets públicos de EEG de sueño descargables sin credenciales.
- "Claude Opus 5.5" para generar el código Python de extracción, transformación y carga (pandas, scikit-learn y sqlite3).
- "Claude Opus 5.5" para generar las sentencias SQL: DDL de la base transaccional y del almacén, ETL `INSERT … SELECT` y consultas OLAP.
- "Claude Opus 5.5" para generar el código del tablero interactivo en Dash/Plotly y validar su paleta de colores para daltonismo.
- "Claude Opus 5.5" para corregir la redacción de las explicaciones y justificaciones de las celdas Markdown.
- "Claude Opus 5.5" para verificar la reproducibilidad ejecutando el cuaderno completo desde cero en un entorno limpio.

Las decisiones metodológicas (elección del dataset, tratamiento de faltantes y atípicos, modelado de los datos y mensaje del tablero) fueron tomadas por mi, que revise manualmente, ejecuté y edité todo el contenido generado.

## Instrucciones de la IA

La IA se usó como asistente en un único hilo de trabajo, con acceso a la carpeta del repositorio y al material de clases. Para reproducir el resultado: dar primero el contexto (paso 0), luego cada instrucción en orden, y revisar y ejecutar la salida de cada paso antes de avanzar. Debajo de cada instrucción se indica la decisión o corrección de la autora (*human-in-the-loop*).

**0. Contexto.**
> Vas a asistirme en el TP final de Introducción a la Ingeniería de Datos (ITBA). Trabajá en `TP-Final/TP-Final_VazquezAylin.ipynb`, con celdas Markdown breves que justifiquen cada decisión (límite de 15 páginas), y ejecutá el cuaderno completo desde cero antes de dar por terminada cada sección.

**1. Dataset.**
> Mi trabajo de tesis del doctorado es sobre procesamiento de señales de sueño, principalmente EEG. Buscá datasets públicos descargables por URL sin login (GitHub, Zenodo, OSF, OpenNeuro, PhysioNet) con variables de alto nivel (estadios por época, edad, sexo, IMC) y no solo señal cruda. Para cada candidato informá licencia, tamaño, faltantes y atípicos reales, y si permite normalizar a 3FN y construir un esquema estrella.

↳ *Decisión que tomé:* Descarté PTB-XL (ECG, ajeno a la tesis), Sleep-EDF (exige procesar señales EDF crudas) y el dataset de EEG psiquiátrico (no es de sueño). Elegí BOAS (OpenNeuro `ds005555`, CC0): EEG de sueño con PSG clínica y vincha wearable registradas en simultáneo. Estoy justamente en el doctorado evaluando algo similar.

**2. Set Up.**
> Creá un `requirements.txt` propio del TP con cotas mínimas (`>=`) para no forzar reinstalaciones en Colab, y una sección *Set Up* que lo instale (copia local o descargada de GitHub) e importe las librerías agrupadas por función, con una semilla fija.

**3. Extracción.**
> Descargá `participants.tsv` y los 256 `events.tsv` desde la réplica de OpenNeuro en GitHub fijando el tag `1.1.3`, en paralelo y con caché en `data/raw`. Cargalos optimizando parámetros: proyección de columnas (`usecols`), tipado reducido (`Int8`, `Int16`, `category`, `float32`) y `na_values=["n/a"]` (convención BIDS). Antes de descartar una columna, verificá con `assert` que es redundante. Compará la memoria de la carga sin optimizar contra la optimizada. Uní PSG y vincha en una "*One Big Table*" validando la relación 1:1, agregá controles de integridad y un diagnóstico que solo identifique faltantes y atípicos.

↳ *Corrección y decisión que tomé:* Pedí aclarar que la reducción de memoria combina proyección de columnas, tipado reducido y unión de dispositivos. En el diagnóstico de faltantes y atípicos busqué y verifiqué valores típicos de PSG (desconexión, artefactos, fases); lo analicé manualmente y brevemente para chequear las condiciones de la base de datos.

**4. Transformación.**
> Convertí los códigos `8` (desconexión) y `-2` (artefacto) en faltantes explícitos sin imputar estadios. Para edad e IMC, buscá primero una imputación deductiva por `pid` antes de una estadística. Derivá el grupo etario y la categoría de IMC (OMS) y calculá indicadores por noche: tiempo total de sueño, eficiencia, latencias, WASO, % de artefactos y kappa de Cohen de cada IA frente al consenso. Detectá noches atípicas con cercos de Tukey e Isolation Forest sobre `RobustScaler`, distinguí atípico de error y no borres noches, solo marcalas con reglas justificadas.

↳ *Decisión que tomé:* No imputar edad ni IMC (miembro "Sin dato") y marcar como no válidas las noches con menos de 4 h de señal válida, en lugar de usar el cerco de Tukey, que falla con la distribución asimétrica de los artefactos.

**5. Base transaccional.**
> Normalizá la tabla única hasta 3FN explicando el grupo repetitivo (1FN), las dependencias parciales (2FN) y las transitivas (3FN), sin guardar atributos derivables. Escribí el DDL de SQLite con llaves primarias, foráneas (incluida una compuesta), `CHECK`, `NOT NULL` y `PRAGMA foreign_keys = ON`. Cargá todas las tablas en una sola transacción (ACID) y validá la integridad referencial, el rechazo de operaciones inválidas y una reconstrucción de ida y vuelta con `JOIN` y pivot en SQL.

↳ *Decisión que tomé:* Cuando la validación mostró que la cantidad de canales depende de la sesión y no del dispositivo, decidí modelar `channels.tsv` con una tabla puente N:M (`SENSOR`, `SESION_SENSOR`) en lugar de descartar esa información. Realice el diagrama de entidad relación a mano y le pedí a Claude que lo grafique y me guie en mejoras del modelo.

**6. Almacén de datos.**
> Guiame en el diseño del almacén a partir de las siguientes preguntas analíticas: 1. ¿Cuánto coincide cada IA con el consenso de expertos, en total y por estadio de sueño? 2. ¿Cómo varía esa concordancia según la edad, el sexo y el IMC de la persona? 3. ¿En qué momento de la noche falla más cada dispositivo? 4. ¿Cómo es la arquitectura del sueño de cada grupo de personas? (concordancia total, por estadio, por edad, sexo e IMC y por momento de la noche). Haz una comparación en el esquema estrella contra copo de nieve. Definí el grano de cada tabla de hechos, separá medidas aditivas (conteos) de no aditivas (kappa) y usá llaves subrogadas, una dimensión con roles para el estadio y un miembro "Sin dato". Cargalo con SQL desde la base transaccional (`ATTACH` + `INSERT … SELECT`) y validá que los indicadores coincidan con los calculados en pandas.
↳ *Decisiones que tomé:*  Elegí el esquema estrella (constelación) para simplificar las consultas del tablero. Claude propuso un diseño que fuimos modificando en conjunto sobre el diagrama de la estrella. Explicitamente le pedí que haya dos tablas de hechos que comparten DIM_SESION.
Medidas aditivas y no aditivas. Le aclaré que promediar kappa por noche no es el kappa del grupo. Le pedí una desnormalización a propósito: DIM_SESION tiene la persona, los grupos derivados y los sensores aplanados como columnas sí/no. Decidí poner "Sin dato" para edad o IMC faltante y "Sin etiqueta válida" para las épocas con artefacto. Y luego una dimensión con dos roles: DIM_ESTADIO aparece como estadio de referencia y como estadio predicho. Explicité que haya llaves subrogadas en todas las dimensiones. Dejé la normalización completa en la base transaccional.
↳ *Conclusión llegada luego de analizar resultados:* «La vincha wearable estadifica el sueño casi tan bien como el PSG clínico, pero pierde precisión en el sueño profundo (N3)»

**7. Tablero.**
> Construí un tablero en Dash (`jupyter_mode="inline"`) de la siguiente manera: Título como la conclusión (El wearable estadifica el sueño casi tan bien como el PSG, pero pierde precisión en N3), números grandes, atributos preatentivos para destacar N3, poca tinta y una paleta validada para daltonismo. Agregá los siguientes filtros: sexo, grupo etario, IMC, tercio de la noche, e incluir o excluir las noches con menos de 4 h válidas. Colocá: 1) Gráfico de barras PSG contra vincha, con N3 destacado y el resto atenuado, más la anotación. 2) Una matriz de confusión, con selector de dispositivo, mostrando que la vincha confunde N3 con N2. 3) Un gráfico de línea de Kappa por grupo etario.

↳ *Revisión final:* Revisé y edité la redacción de las celdas Markdown y verifiqué que cada sección esté correcta antes de incorporarla al repositorio.

---

## Contexto de investigación

**Dataset elegido:** *The Bitbrain Open Access Sleep (BOAS) dataset* (López-Larraz et al.), OpenNeuro `ds005555` v1.1.3, DOI [10.18112/openneuro.ds005555.v1.1.3](https://doi.org/10.18112/openneuro.ds005555.v1.1.3), licencia CC0.
- 128 noches de 100 personas adultas sanas, registradas en simultáneo con una PSG clínica (Micromed) y una vincha de EEG wearable (Bitbrain).
- Cada época de 30 s tiene tres etiquetas de estadio: el consenso de tres expertos sobre la PSG (criterios AASM), la estadificación automática con IA sobre la PSG y la estadificación automática con IA sobre la vincha.
- Los datos están en formato [BIDS](https://bids-specification.readthedocs.io/en/stable/): `participants.tsv` (edad, sexo, IMC) y un `events.tsv` por noche y dispositivo.

**Pregunta:** ¿Qué tan bien estadifica el sueño un dispositivo wearable de EEG frente al estándar clínico (polisomnografía puntuada por expertos), y cómo varía esa concordancia según el estadio de sueño y las características de la persona (edad, sexo, IMC)?

**Relación con mi tesis:** Mi tesis aborda el procesamiento de señales de sueño, principalmente EEG. Validar dispositivos portátiles contra la polisomnografía (PSG) es un paso necesario para llevar el monitoreo del sueño fuera del laboratorio.

---

## Set Up

Instalación de las dependencias que no vienen preinstaladas en Google Colab y carga de librerías. La lista completa de dependencias está en [`TP-Final/requirements.txt`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/requirements.txt): si el archivo está disponible localmente (ejecución desde el repositorio clonado) se usa esa copia; si no (Colab), se descarga desde GitHub.

Para que el cuaderno sea legible, el código más extenso está en la carpeta [`TP-Final/src/`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src), que se obtiene de la misma forma (copia local o descarga desde GitHub). Cada archivo tiene un encabezado que explica su contenido:

| Archivo | Qué hace | Se usa en |
|---|---|---|
| `oltp_esquema.sql` | Crea las 9 tablas de la base transaccional con sus llaves primarias, foráneas y restricciones. | 3.3 |
| `dw_esquema.sql` | Crea las dimensiones y las tablas de hechos del almacén en estrella. | 4.1 |
| `dw_etl.sql` | Carga el almacén desde la base transaccional en una única transacción: dimensiones, cubo de concordancia, vista de κ por noche y hechos por noche. | 4.2 |
| `dw_cubo.sql` | Consulta del cubo de concordancia unido a sus dimensiones. | 4.4 y 5 |
| `metricas.py` | `resumir_noche()`: indicadores de cada noche. `concordancia()`: acuerdo y κ de Cohen a partir de conteos. | 2.3, 4.4 y 5 |
| `tablero.py` | Gráficos, tarjetas, filtros y diseño del tablero en Dash. | 5 |

In [ ]:
# Instalación de dependencias y obtención del código auxiliar (src/)
import os, sys, urllib.request
from pathlib import Path

URL_REPO = "https://raw.githubusercontent.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/main/TP-Final"
ARCHIVOS_SRC = ["oltp_esquema.sql", "dw_esquema.sql", "dw_etl.sql", "dw_cubo.sql", "metricas.py", "tablero.py"]
DIR_SRC = Path("src")

def obtener(ruta: str) -> None:
    """Usa la copia local si existe (repositorio clonado); si no (Colab), la descarga de GitHub."""
    if not os.path.exists(ruta):
        Path(ruta).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{URL_REPO}/{ruta}", ruta)

obtener("requirements.txt")
for archivo in ARCHIVOS_SRC:
    obtener(f"src/{archivo}")
sys.path.insert(0, str(DIR_SRC.resolve()))

%pip install -q -r requirements.txt

In [ ]:
# Importación de librerías
# Librería estándar
import sqlite3
from concurrent.futures import ThreadPoolExecutor

# Ingesta y manipulación
import numpy as np
import pandas as pd
import requests

# Análisis
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# Código auxiliar del TP (src/)
from metricas import resumir_noche, concordancia
from tablero import crear_app

def leer_sql(archivo: str, **parametros) -> str:
    """Lee un archivo SQL de src/ y completa sus parámetros {nombre}, si los tiene."""
    sql = (DIR_SRC / archivo).read_text(encoding="utf-8")
    return sql.format(**parametros) if parametros else sql

# Configuración general
SEED = 42                                   # Semilla para reproducibilidad
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 2)   # float32 se muestra sin ruido de redondeo
EN_COLAB = "google.colab" in str(get_ipython())

import sklearn, plotly, dash
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'} | " +
      " | ".join(f"{lib.__name__} {lib.__version__}" for lib in (pd, np, sklearn, plotly, dash)))

---

## 1. Extracción

**Fuente.** Los archivos se descargan desde la réplica oficial de OpenNeuro en GitHub ([`OpenNeuroDatasets/ds005555`](https://github.com/OpenNeuroDatasets/ds005555)), fijando el *tag* `1.1.3` en lugar de la rama `main`: cualquier ejecución futura lee exactamente la misma versión publicada, aunque el dataset se actualice.

**Alcance.** Se descargan `participants.tsv` (una fila por noche registrada) y los 256 `events.tsv` (128 noches × 2 dispositivos), que contienen la estadificación por época de 30 s. Las señales crudas (`.edf`, ~180 MB por noche) **no** se descargan porque la pregunta de investigación se responde con las etiquetas de estadio.

**Caché.** Cada archivo se guarda en `data/raw/ds005555/`; si ya existe, no se vuelve a descargar.

In [3]:
# Parámetros de la fuente y estructura de carpetas
DATASET = "ds005555"
VERSION = "1.1.3"   # tag de la versión publicada en OpenNeuro
BASE_URL = f"https://raw.githubusercontent.com/OpenNeuroDatasets/{DATASET}/{VERSION}"

DIR_RAW = Path("data/raw") / DATASET
DIR_PROCESSED = Path("data/processed")
for carpeta in (DIR_RAW, DIR_PROCESSED):
    carpeta.mkdir(parents=True, exist_ok=True)

def descargar(ruta_relativa: str) -> Path:
    """Descarga un archivo del dataset si no está en caché y devuelve su ruta local."""
    destino = DIR_RAW / ruta_relativa
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        respuesta = requests.get(f"{BASE_URL}/{ruta_relativa}", timeout=30)
        respuesta.raise_for_status()  # Falla explícitamente ante un error HTTP
        destino.write_bytes(respuesta.content)
    return destino

def memoria_mb(df: pd.DataFrame) -> float:
    """Memoria RAM ocupada por un DataFrame (incluye el contenido de los strings)."""
    return df.memory_usage(deep=True).sum() / 1024**2

### 1.1 Participantes (`participants.tsv`)

Por convención BIDS, `participant_id` identifica a cada **noche registrada** (`sub-1` … `sub-128`), mientras que `pid` identifica a la **persona** (algunas personas durmieron más de una noche). Los parámetros de carga se eligen a partir del diccionario de datos (`participants.json`):

| Columna | Tipo | Justificación |
|---|---|---|
| `participant_id` | `string` | Identificador textual de la sesión. |
| `pid` | `Int16` | Entero nullable; 100 personas. |
| `age` | `Int8` | Edad en años (18–82) cabe en 8 bits; *nullable* para admitir faltantes. |
| `sex` | `category` | Solo dos niveles (`F`, `M`). |
| `bmi` | `float32` | Body-mass-index de la persona. |

`na_values=["n/a"]` interpreta explícitamente el marcador de faltante que usa BIDS.

In [ ]:
descargar("participants.json")  # Diccionario de variables (metadatos)

# Descarga y carga de los datos de participantes
participantes = pd.read_csv(
    descargar("participants.tsv"),
    sep="\t",
    usecols=["participant_id", "pid", "age", "sex", "bmi"],
    dtype={"participant_id": "string", "pid": "Int16", "age": "Int8",
           "sex": "category", "bmi": "float32"},
    na_values=["n/a"],
)
print(f"{len(participantes)} noches registradas | {participantes['pid'].nunique()} personas | "
      f"{memoria_mb(participantes):.3f} MB")

### 1.2 Hipnogramas (`events.tsv`)

Cada noche tiene un archivo por dispositivo: `acq-psg` (polisomnografía) y `acq-headband` (vincha wearable). Ambos tienen una fila por época de 30 s con las columnas `onset`, `duration`, `begsample`, `endsample`, `offset` y las etiquetas de estadio (`stage_hum`, solo en PSG; `stage_ai`). Las descargas se hacen en paralelo porque son operaciones de entrada/salida.

In [5]:
SESIONES = participantes["participant_id"].tolist()
ADQUISICIONES = ["psg", "headband"]

def ruta_eventos(sesion: str, acq: str) -> str:
    return f"{sesion}/eeg/{sesion}_task-Sleep_acq-{acq}_events.tsv"

with ThreadPoolExecutor(max_workers=8) as pool:
    archivos = list(pool.map(descargar, [ruta_eventos(s, a) for s in SESIONES for a in ADQUISICIONES]))
print(f"{len(archivos)} archivos de eventos disponibles en {DIR_RAW}/")

256 archivos de eventos disponibles en data/raw/ds005555/


Primero se cargan los archivos con la configuración por defecto de pandas para verificar que las columnas que se van a descartar son redundantes:

- `duration` vale siempre 30 s y `offset` siempre 0 (constantes, sin información).
- `begsample` y `endsample` se deducen de `onset` y de la frecuencia de muestreo (256 Hz, declarada en los `*_eeg.json`).

Luego se cargan solo `onset` y las etiquetas, con enteros de 8 y 32 bits.

La reducción de memoria entre ambas cargas combina tres efectos:
1. **Proyección de columnas:** de 7 columnas a 3 (PSG) o 2 (vincha).
2. **Tipado reducido:** `int32` para `onset` y `Int8` (entero que admite nulos) para los estadios, en lugar de `int64`.
3. **Unión de dispositivos:** al unir PSG y vincha por época, `onset` se almacena una sola vez y las filas se reducen a la mitad.

Los dos primeros corresponden a la optimización de parámetros de carga; el tercero es una consecuencia de reorganizar la tabla.

In [7]:
FS = 256  # Hz, frecuencia de muestreo declarada en cada *_eeg.json

In [10]:
def leer_eventos(sesion: str, acq: str, optimizado: bool = True) -> pd.DataFrame:
    ruta = DIR_RAW / ruta_eventos(sesion, acq)
    if not optimizado:
        return pd.read_csv(ruta, sep="\t")
    columnas = ["onset", "stage_hum", "stage_ai"] if acq == "psg" else ["onset", "stage_ai"]
    return pd.read_csv(ruta, sep="\t", usecols=columnas, na_values=["n/a"],
                       dtype={"onset": "int32", "stage_hum": "Int8", "stage_ai": "Int8"})

# Carga por separado: todas las columnas con tipos por defecto (int64)
por_separado = pd.concat({(s, a): leer_eventos(s, a, optimizado=False)
                     for s in SESIONES for a in ADQUISICIONES},
                    names=["participant_id", "acq", "fila"])

# Validación de las columnas que se descartan
assert (por_separado["duration"] == 30).all()
assert (por_separado["offset"] == 0).all()
assert (por_separado["begsample"] == por_separado["onset"] * FS + 1).all()
assert (por_separado["endsample"] == (por_separado["onset"] + 30) * FS).all()
print("✓ duration, offset, begsample y endsample son redundantes con onset")

✓ duration, offset, begsample y endsample son redundantes con onset


In [11]:
# Carga optimizada: proyección de columnas + tipos reducidos
def cargar_dispositivo(acq: str) -> pd.DataFrame:
    df = pd.concat({s: leer_eventos(s, acq) for s in SESIONES}, names=["participant_id", "fila"])
    return df.reset_index(level="participant_id").reset_index(drop=True)

psg = cargar_dispositivo("psg").rename(columns={"stage_ai": "stage_ai_psg"})
vincha = cargar_dispositivo("headband").rename(columns={"stage_ai": "stage_ai_hb"})

# Unión época a época: cada época debe existir en ambos dispositivos (relación 1:1)
eventos = psg.merge(vincha, on=["participant_id", "onset"], how="outer",
                    validate="one_to_one", indicator=True)
assert (eventos["_merge"] == "both").all(), "Hay épocas presentes en un solo dispositivo"
eventos = eventos.drop(columns="_merge")
eventos["participant_id"] = eventos["participant_id"].astype("category")

print(f"Carga por separado   : {memoria_mb(por_separado):6.2f} MB ({len(por_separado):,} filas, PSG y vincha por separado)")
print(f"Carga optimizada: {memoria_mb(eventos):6.2f} MB ({len(eventos):,} épocas, PSG y vincha unidos)")
del por_separado

Carga por separado   :  13.98 MB (240,190 filas, PSG y vincha por separado)
Carga optimizada:   1.38 MB (120,095 épocas, PSG y vincha unidos)


### 1.3 Tabla única de extracción y controles de integridad

Se unen las épocas con los atributos de cada noche en una única tabla plana (*One Big Table*), que es el punto de partida de la Transformación. Antes de avanzar se verifican las reglas de integridad que el resto del pipeline asume.

In [ ]:
obt = eventos.merge(participantes.astype({"participant_id": "category"}),
                    on="participant_id", how="left", validate="many_to_one")

# Controles de integridad
assert obt["participant_id"].nunique() == 128 and obt["pid"].nunique() == 100
assert obt.groupby("participant_id", observed=True)["onset"].diff().dropna().eq(30).all(), \
    "Hay saltos entre épocas consecutivas"
assert set(obt["stage_hum"].unique()) <= {0, 1, 2, 3, 4, 8}
assert set(obt["stage_ai_psg"].unique()) | set(obt["stage_ai_hb"].unique()) <= {-2, 0, 1, 2, 3, 4}
print("✓ 128 noches, 100 personas, épocas contiguas y códigos de estadio dentro del dominio")

obt.to_parquet(DIR_PROCESSED / "01_extraccion_obt.parquet", index=False)
print(f"OBT: {obt.shape[0]:,} filas × {obt.shape[1]} columnas | {memoria_mb(obt):.2f} MB")

### 1.4 Primer diagnóstico de calidad

En esta etapa solo se identifican los problemas; su resolución se justifica en la sección de Transformación.

In [13]:
CODIGOS = {"stage_hum": 8, "stage_ai_psg": -2, "stage_ai_hb": -2}  # 8 = desconexión del PSG, -2 = artefacto
print("Faltantes en participants.tsv:")
print(participantes[participantes.isna().any(axis=1)].to_string(index=False))

print("\nÉpocas con código especial (no son un estadio de sueño):")
for col, cod in CODIGOS.items():
    n = (obt[col] == cod).sum()
    print(f"  {col:<13} = {cod:>2}: {n:>5,} épocas ({n / len(obt):.1%})")

horas = obt.groupby("participant_id", observed=True).size() * 30 / 3600
print(f"\nDuración del registro por noche (h): mediana {horas.median():.1f} | "
      f"mín {horas.min():.1f} ({horas.idxmin()}) | máx {horas.max():.1f}")

primer_estadio = obt.groupby("participant_id", observed=True)["stage_hum"].first()
print(f"Noches que no comienzan en vigilia: {primer_estadio[primer_estadio != 0].to_dict()}")

por_persona = participantes.groupby("pid")[["age", "bmi"]].nunique()
print("\nPersonas con más de un valor de edad o IMC entre noches:")
print(participantes[participantes["pid"].isin(por_persona[(por_persona > 1).any(axis=1)].index)]
      .to_string(index=False))

Faltantes en participants.tsv:
participant_id  age sex  bmi  pid
        sub-16 <NA>   F 26.8   11
        sub-29   23   F  NaN   21

Épocas con código especial (no son un estadio de sueño):
  stage_hum     =  8:   336 épocas (0.3%)
  stage_ai_psg  = -2:   550 épocas (0.5%)
  stage_ai_hb   = -2: 4,881 épocas (4.1%)

Duración del registro por noche (h): mediana 7.8 | mín 3.4 (sub-8) | máx 9.3
Noches que no comienzan en vigilia: {'sub-1': 3}

Personas con más de un valor de edad o IMC entre noches:
participant_id  age sex  bmi  pid
        sub-23   34   M 28.1   15
       sub-124   35   M 27.3   15


**Hallazgos** (que se resuelven en la Transformación):

1. **Faltantes:** Una noche sin edad (`sub-16`) y una sin IMC (`sub-29`).
2. **Códigos especiales:** Las etiquetas `8` (desconexión del PSG, p. ej. para ir al baño) y `-2` (artefacto o dato faltante en la IA) no son estadios de sueño. Son NAs codificados como números y se tratan como faltantes. La vincha concentra la mayor proporción de artefactos.
3. **Atípico de duración:** `sub-8` registra 3,4 h, muy por debajo de la mediana (~7,8 h).
4. **Inicio de registro:** `sub-1` comienza directamente en N3 (las otras 127 noches comienzan en vigilia), lo que indica que el registro empezó con la persona ya dormida; su latencia de sueño no es calculable.
5. **Dependencias:** El sexo es constante por persona, pero la edad y el IMC pueden cambiar entre noches de la misma persona (`pid` 15). Esto guía la normalización: el sexo es atributo de la persona; la edad y el IMC, de la noche.

---

## 2. Transformación

***Objetivo:*** Resolver los problemas detectados en el diagnóstico y derivar las variables que necesitan los modelos transaccional y dimensional. El criterio general tomado es que no se elimina información cruda. Los valores inválidos se convierten en faltantes explícitos y las noches problemáticas se marcan con reglas documentadas, para que cada análisis decida qué incluir.

### 2.1 Códigos especiales → Faltantes y variables temporales

Los códigos `8` (desconexión del PSG) y `-2` (artefacto) son faltantes codificados como números: si se dejaran, cualquier promedio o conteo por estadio los trataría como un estadio más. Se reemplazan por `NA`. No se imputan: asignar un estadio a una época sin señal equivale a inventar una etiqueta clínica, y la propia tasa de faltantes es un indicador de calidad del dispositivo que conviene conservar.

Se agregan además el número de época, la hora desde el inicio del registro y el tercio de la noche (contexto temporal para el almacén).

In [20]:
COLS_ESTADIO = ["stage_hum", "stage_ai_psg", "stage_ai_hb"]
epocas = obt[["participant_id", "onset"] + COLS_ESTADIO].copy()
for col, codigo in CODIGOS.items():          # CODIGOS se definió en el diagnóstico (1.4)
    epocas[col] = epocas[col].mask(epocas[col] == codigo)

epocas["epoca"] = (epocas["onset"] // 30).astype("int16")
epocas["hora"] = (epocas["onset"] // 3600).astype("int8")
n_epocas = epocas.groupby("participant_id", observed=True)["epoca"].transform("size")
epocas["tercio"] = (epocas["epoca"] * 3 // n_epocas + 1).astype("int8")

print("Faltantes explícitos por columna de estadio:")
print(epocas[COLS_ESTADIO].isna().sum().to_string())

Faltantes explícitos por columna de estadio:
stage_hum        336
stage_ai_psg     550
stage_ai_hb     4881


### 2.2 Faltantes en edad e IMC

Antes de considerar una imputación estadística se busca una imputación deductiva ya que si la misma persona (`pid`) tuviera otra noche con el dato informado, se podría completar sin error. Pero este es el caso porque ambas personas `sub-16` y `sub-29` registraron una sola noche.

**Decisión: *no imputar*.** Son 2 valores en 256 (0.8% por variable) y el mecanismo es compatible con una omisión al azar en un dato autorreportado. Edad e IMC se usan para agrupar noches, no como medidas: imputar por mediana asignaría a la persona a un grupo de forma arbitraria. Decido conservarlo como `NULL` en la base transaccional y como categoría "Sin dato" en el almacén (lo cual es una práctica estándar del modelado dimensional para miembros desconocidos).

Se derivan el grupo etario y la categoría de IMC según los cortes de la OMS.

In [21]:
noches = participantes.copy()
faltantes = noches.loc[noches[["age", "bmi"]].isna().any(axis=1), "pid"]
otras_noches = noches[noches["pid"].isin(faltantes)].groupby("pid").size()
print(f"Noches registradas por las personas con faltantes: { {int(k): int(v) for k, v in otras_noches.items()} }")  # 1 → no hay deducción posible

noches["grupo_etario"] = pd.cut(noches["age"], bins=[17, 30, 45, 60, 120],
                                labels=["18-30", "31-45", "46-60", ">60"])
noches["categoria_imc"] = pd.cut(noches["bmi"], bins=[0, 18.5, 25, 30, 100], right=False,
                                 labels=["Bajo peso", "Normal", "Sobrepeso", "Obesidad"])
for col in ["grupo_etario", "categoria_imc"]:
    print(f"{col}: {noches[col].value_counts(sort=False, dropna=False).to_dict()}")

Noches registradas por las personas con faltantes: {11: 1, 21: 1}
grupo_etario: {'18-30': 55, '31-45': 27, '46-60': 10, '>60': 35, nan: 1}
categoria_imc: {'Bajo peso': 1, 'Normal': 81, 'Sobrepeso': 38, 'Obesidad': 7, nan: 1}


### 2.3 Indicadores por noche

La función `resumir_noche()` de [`src/metricas.py`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/metricas.py) calcula estos indicadores para cada noche a partir de sus épocas.

Para detectar noches atípicas se resume cada noche con indicadores de arquitectura del sueño (calculados sobre el consenso humano, que es el estándar de referencia), de calidad de señal y de concordancia:

| Indicador | Definición |
|---|---|
| `horas_registro` | Duración total del registro. |
| `horas_validas_psg` / `_hb` | Horas con etiqueta válida del consenso y de la IA de cada dispositivo. |
| `tst_min` | Tiempo total de sueño (N1 + N2 + N3 + REM). |
| `eficiencia` | `tst_min` / tiempo con etiqueta válida. El dataset no informa la hora de apagado de luces, así que se usa todo el registro. |
| `latencia_sueno_min` / `latencia_rem_min` | Desde el inicio del registro hasta la primera época de sueño, y desde ésta hasta la primera de REM. |
| `waso_min` | Vigilia entre la primera y la última época de sueño. |
| `pct_artefacto_hb` | % de épocas sin etiqueta de la vincha. |
| `kappa_psg` / `kappa_hb` | κ de Cohen entre cada IA y el consenso humano (épocas válidas en ambos). |

In [ ]:
indicadores = epocas.groupby("participant_id", observed=True).apply(resumir_noche)
mediana = indicadores.median()
print(f"Indicadores de {len(indicadores)} noches. Medianas: TST {mediana['tst_min'] / 60:.1f} h | "
      f"eficiencia {mediana['eficiencia']:.0%} | κ IA-PSG {mediana['kappa_psg']:.2f} | κ IA-vincha {mediana['kappa_hb']:.2f}")

### 2.4 Detección de noches atípicas

Aca se combinan dos enfoques vistos en la materia: **cercos de Tukey** (univariado, 1,5 × IQR) e **Isolation Forest** (multivariado, sobre los indicadores escalados con `RobustScaler`, que es resistente a los propios atípicos).

In [ ]:
VARS_ATIPICOS = ["horas_registro", "tst_min", "eficiencia", "pct_artefacto_hb", "kappa_psg", "kappa_hb"]

def fuera_de_cercos(x: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = x.quantile([0.25, 0.75])
    return (x < q1 - k * (q3 - q1)) | (x > q3 + k * (q3 - q1))

tukey = indicadores[VARS_ATIPICOS].apply(fuera_de_cercos)
print("Noches fuera de los cercos de Tukey por indicador:")
for col in VARS_ATIPICOS:
    print(f"  {col:<17} {tukey[col].sum():>2} noches")

X = RobustScaler().fit_transform(indicadores[VARS_ATIPICOS])
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED).fit(X)
atipicas_if = indicadores.index[iso.predict(X) == -1]
print(f"\nIsolation Forest (5 %): {sorted(atipicas_if, key=lambda s: int(s[4:]))}")

### 2.5 Decisiones tomadas sobre los atípicos

1. **Baja concordancia o mal dormir no son errores.** Las noches con κ (kappa) bajo o con poco tiempo de sueño (p. ej. `sub-4`, `sub-10`, `sub-121`) son justamente el fenómeno que se quiere estudiar. Eliminarlas sobreestimaría el desempeño de los algoritmos. **Decisión: *Se conservan*.**
2. **Criterio de señal mínima: al menos 4 h válidas.** Un κ calculado sobre pocas horas no es comparable con el de una noche completa, y los artefactos no ocurren al azar (se concentran en los movimientos y vigilias). Los cercos de Tukey no sirven para esta variable: `pct_artefacto_hb` es una variable muy asimétrica (mediana 0.2%), así que el cerco superior queda en ~8% y marca 17 noches con señal de sobra. Tukey supone una distribución aproximadamente simétrica. Por eso se usa un criterio absoluto: una noche es válida para un dispositivo solo si tiene ≥ 4 h de épocas válidas (`valida_psg` y `valida_hb`).
   - `sub-8` (registro de 3,4 h) queda excluida en ambos dispositivos.
   - `sub-1` (2,0 h válidas) y `sub-13` (3,8 h) quedan excluidas solo para la vincha.
   - `sub-53` (registro corto, 5,5 h) aparece como atípica en ambos métodos, pero supera el umbral y se conserva.
3. **Latencia no calculable.** `sub-1` comienza dormido, así que su latencia de sueño queda como `NA`.
4. **Ninguna noche se borra.** Las marcas viajan a la base transaccional y al almacén, donde el tablero las usa como filtro.

In [23]:
HORAS_MINIMAS = 4
indicadores["valida_psg"] = indicadores["horas_validas_psg"] >= HORAS_MINIMAS
indicadores["valida_hb"] = indicadores["horas_validas_hb"] >= HORAS_MINIMAS
print("Noches no válidas:",
      {d: sorted(indicadores.index[~indicadores[f"valida_{d}"]]) for d in ["psg", "hb"]})

noches = noches.merge(indicadores[["valida_psg", "valida_hb"]], left_on="participant_id",
                      right_index=True, how="left", validate="one_to_one")

epocas.to_parquet(DIR_PROCESSED / "02_epocas_limpias.parquet", index=False)
noches.to_parquet(DIR_PROCESSED / "02_noches.parquet", index=False)
print(f"Épocas: {epocas.shape} | Noches: {noches.shape} → guardadas en {DIR_PROCESSED}/")

Noches no válidas: {'psg': ['sub-8'], 'hb': ['sub-1', 'sub-13', 'sub-8']}
Épocas: (120095, 8) | Noches: (128, 9) → guardadas en data/processed/


---

## 3. Carga en la base de datos transaccional (SQLite)

La base transaccional (OLTP) es el sistema de registro del pipeline: Guarda cada dato una sola vez, garantiza la integridad referencial mediante llaves primarias y foráneas, y valida las reglas de negocio con restricciones (`CHECK`, `NOT NULL`). Las cargas se hacen dentro de transacciones que cumplen las propiedades **ACID**. A partir de esta base se construye después el almacén de datos.

### 3.1 Normalización

Se parte de la tabla única de la Transformación, cuya llave es (`participant_id`, `onset`):

| Forma normal | Problema en la tabla única | Acción |
|---|---|---|
| **1FN** | `stage_hum`, `stage_ai_psg` y `stage_ai_hb` son un **grupo repetitivo**: el mismo atributo (estadio) registrado por tres evaluadores distintos. | Se pasan a filas: una etiqueta por (sesión, época, evaluador). La llave compuesta pasa a ser {`sesion_id`, `nro_epoca`, `evaluador_id`}. |
| **2FN** | **Dependencias parciales**: edad, IMC, `pid` y las marcas de validez dependen solo de `sesion_id` (una parte de la llave); `onset_s`, de (`sesion_id`, `nro_epoca`); la descripción del evaluador, solo de `evaluador_id`. | Se extraen `SESION`, `EPOCA` y `EVALUADOR`. |
| **3FN** | **Dependencias transitivas**: `sesion_id → pid → sexo`; `evaluador_id → dispositivo → fabricante, modelo, fs`; `estadio_id → sigla, nombre`. | Se extraen `PERSONA`, `DISPOSITIVO` y `ESTADIO`. |

La configuración de sensores de cada noche (`channels.tsv`) también viene en formato ancho: una columna 0/1 por grupo de sensores, que es otro **grupo repetitivo**. Se lleva a filas y queda una relación **N:M** entre sesiones y sensores (una noche usa varios sensores y cada sensor se usa en muchas noches), que se resuelve con la tabla puente `SESION_SENSOR`. La descripción de cada sensor depende solo del sensor, así que va en su propia tabla `SENSOR` (3FN).

El grupo etario y la categoría de IMC no se guardan en la base transaccional: se deducen de la edad y del IMC (otra dependencia transitiva) y se calculan recién en el almacén.

### 3.2 Diagrama entidad-relación

![Diagrama entidad-relación de la base transaccional](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/img/er_oltp.png?raw=true)

- Una **persona** puede tener varias **sesiones** (noches); cada **sesion** tiene muchas **epocas** de 30s de duración.
- Cada **epoca** recibe una **etiqueta** por **evaluador** (consenso humano, IA sobre PSG e IA sobre vincha). _Esta tabla resuelve la relación N:M entre épocas y evaluadores._
- Cada **sesion** usa varios **sensores** y cada **sensor** pertenece a un **dispositivo**. _La tabla puente `SESION_SENSOR` resuelve la relación N:M entre sesiones y sensores._
- Cada **evaluador** trabaja sobre la señal de un **dispositivo**. La **etiqueta** referencia a un **estadio**, o queda en `NULL` si la época no tiene etiqueta válida.

In [31]:
rutas_json = [f"{s}/eeg/{s}_task-Sleep_acq-{a}_eeg.json" for s in SESIONES for a in ADQUISICIONES]
with ThreadPoolExecutor(max_workers=8) as pool:
    metadatos = [pd.read_json(ruta, typ="series") for ruta in pool.map(descargar, rutas_json)]

meta = pd.DataFrame(metadatos).assign(acq=[a for _ in SESIONES for a in ADQUISICIONES])
# EEGChannelCount se excluye: en el PSG varía entre noches (8 a 13 canales), depende de la sesión
canales = meta.groupby("acq")["EEGChannelCount"].unique()
print("Cantidad de canales observada por dispositivo:", {k: sorted(map(int, v)) for k, v in canales.items()})
COLS_DISP = ["Manufacturer", "ManufacturersModelName", "SamplingFrequency"]
assert (meta.groupby("acq")[COLS_DISP].nunique() == 1).all().all(), "Los metadatos varían entre sesiones"

dispositivo = (meta.drop_duplicates("acq")[["acq"] + COLS_DISP]
               .rename(columns={"Manufacturer": "fabricante", "ManufacturersModelName": "modelo",
                                "SamplingFrequency": "fs_hz"})
               .reset_index(drop=True))
dispositivo.insert(0, "dispositivo_id", dispositivo.index + 1)
dispositivo

Cantidad de canales observada por dispositivo: {'headband': [2], 'psg': [8, 9, 11, 12, 13]}


,dispositivo_id,acq,fabricante,modelo,fs_hz
0,1,psg,Micromed,SD plus,256
1,2,headband,Bitbrain Technologies,IKON Sleep,256


Los atributos de cada dispositivo se leen de los metadatos BIDS (`*_eeg.json`). Antes de usarlos se verifica que sean idénticos en las 128 noches, es decir, que dependen del dispositivo y no de la sesión. La verificación excluye `EEGChannelCount` ya que en el PSG la cantidad de canales cambia según los sensores colocados cada noche (de 8 a 13), así que depende de la sesión y no se guarda en `DISPOSITIVO`. Esa variación queda representada en la tabla puente `SESION_SENSOR`.

**Sensores por sesión (`channels.tsv`).** El archivo tiene una fila por noche y una columna 0/1 por grupo de sensores. Su diccionario (`channels.json`) describe cada grupo. El prefijo del código (`PSG_` o `HB_`) indica a qué dispositivo pertenece el sensor.

In [32]:
descripcion_sensores = pd.read_json(descargar("channels.json"), typ="series")
COLS_SENSOR = [c for c in descripcion_sensores.index if c != "REC"]
canales = pd.read_csv(descargar("channels.tsv"), sep="\t", na_values=["n/a"],
                      dtype={"REC": "string", **{c: "Int8" for c in COLS_SENSOR}})

assert set(canales["REC"]) == set(SESIONES), "channels.tsv no coincide con las sesiones"
assert canales[COLS_SENSOR].isin([0, 1]).all().all(), "Valores fuera del dominio {0, 1}"

ACQ_POR_PREFIJO = {"PSG": "psg", "HB": "headband"}
sensor = pd.DataFrame({
    "sensor_id": range(1, len(COLS_SENSOR) + 1),
    "codigo": COLS_SENSOR,
    "dispositivo_id": [dispositivo.set_index("acq").loc[ACQ_POR_PREFIJO[c.split("_")[0]], "dispositivo_id"]
                       for c in COLS_SENSOR],
    "descripcion": [descripcion_sensores[c]["Description"] for c in COLS_SENSOR],
})

# Formato ancho → largo: una fila por (sesión, sensor presente)
sesion_sensor = (canales.melt(id_vars="REC", var_name="codigo", value_name="presente")
                 .query("presente == 1")
                 .merge(sensor[["codigo", "sensor_id"]], on="codigo")
                 .rename(columns={"REC": "sesion_id"})[["sesion_id", "sensor_id"]])

print(f"{len(sensor)} sensores | {len(sesion_sensor)} filas en la tabla puente | "
      f"{canales[COLS_SENSOR].drop_duplicates().shape[0]} configuraciones distintas de sensores")
print("Noches con cada sensor:", {c: int(n) for c, n in canales[COLS_SENSOR].sum().items()})

10 sensores | 965 filas en la tabla puente | 9 configuraciones distintas de sensores
Noches con cada sensor: {'PSG_EEG': 128, 'PSG_EOG': 128, 'PSG_EMG': 128, 'PSG_BELTS': 62, 'PSG_THER': 62, 'PSG_CAN': 19, 'PSG_PPG': 104, 'HB_EEG': 128, 'HB_IMU': 94, 'HB_PULSE': 112}


### 3.3 Esquema (DDL)

Las sentencias `CREATE TABLE` de las 9 tablas están en [`src/oltp_esquema.sql`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/oltp_esquema.sql).

Cada tabla declara su llave primaria, sus llaves foráneas y las restricciones de dominio. `PRAGMA foreign_keys = ON` es necesario porque SQLite no aplica la integridad referencial por defecto. La base se recrea en cada ejecución para que el cuaderno sea reproducible.

In [ ]:
DIR_DB = Path("data/db")
DIR_DB.mkdir(parents=True, exist_ok=True)
RUTA_OLTP = DIR_DB / "boas_oltp.db"
RUTA_OLTP.unlink(missing_ok=True)

DDL_OLTP = leer_sql("oltp_esquema.sql")   # 9 tablas con PK, FK, CHECK y NOT NULL

conn_oltp = sqlite3.connect(RUTA_OLTP)
conn_oltp.execute("PRAGMA foreign_keys = ON")
conn_oltp.executescript(DDL_OLTP)
print("Tablas creadas:", [t for (t,) in conn_oltp.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY rowid")])

### 3.4 Carga

Se arma un DataFrame por tabla y se insertan todos en una sola transacción (`with conn:`). Si cualquier fila viola una restricción, se revierte la carga completa y la base nunca queda a medio cargar (atomicidad y consistencia). El orden de inserción respeta las llaves foráneas: primero las tablas referenciadas.

In [34]:
# PERSONA: el sexo debe ser único por persona (verificación de la dependencia pid → sexo)
persona = noches[["pid", "sex"]].drop_duplicates().rename(columns={"sex": "sexo"})
assert persona["pid"].is_unique

sesion = (noches.rename(columns={"participant_id": "sesion_id", "age": "edad", "bmi": "imc"})
          [["sesion_id", "pid", "edad", "imc", "valida_psg", "valida_hb"]]
          .assign(imc=lambda d: d["imc"].astype("float64").round(1)))

evaluador = pd.DataFrame({
    "evaluador_id": [1, 2, 3],
    "descripcion": ["Consenso de 3 expertos (AASM)", "IA sobre PSG", "IA sobre vincha"],
    "tipo": ["humano", "IA", "IA"],
    "dispositivo_id": dispositivo.set_index("acq").loc[["psg", "psg", "headband"], "dispositivo_id"].to_list(),
})
estadio = pd.DataFrame({
    "estadio_id": [0, 1, 2, 3, 4],
    "sigla": ["W", "N1", "N2", "N3", "REM"],
    "nombre": ["Vigilia", "Sueño NREM 1", "Sueño NREM 2", "Sueño NREM 3 (profundo)", "Sueño REM"],
    "es_sueno": [0, 1, 1, 1, 1],
})
epoca = epocas[["participant_id", "epoca", "onset"]].rename(
    columns={"participant_id": "sesion_id", "epoca": "nro_epoca", "onset": "onset_s"})

# ETIQUETA: el grupo repetitivo de 3 columnas pasa a filas (1FN)
EVALUADOR_POR_COLUMNA = {"stage_hum": 1, "stage_ai_psg": 2, "stage_ai_hb": 3}
etiqueta = (epocas.melt(id_vars=["participant_id", "epoca"], value_vars=COLS_ESTADIO,
                        var_name="columna", value_name="estadio_id")
            .assign(evaluador_id=lambda d: d["columna"].map(EVALUADOR_POR_COLUMNA))
            .rename(columns={"participant_id": "sesion_id", "epoca": "nro_epoca"})
            [["sesion_id", "nro_epoca", "evaluador_id", "estadio_id"]])

def insertar(conn: sqlite3.Connection, tabla: str, df: pd.DataFrame) -> None:
    """Inserta un DataFrame convirtiendo los faltantes (NA/NaN) en NULL."""
    columnas = [[None if pd.isna(v) else v for v in df[c].tolist()] for c in df.columns]
    sql = f"INSERT INTO {tabla} ({', '.join(df.columns)}) VALUES ({', '.join('?' * df.shape[1])})"
    conn.executemany(sql, zip(*columnas))

TABLAS_OLTP = {"persona": persona, "sesion": sesion, "dispositivo": dispositivo,
               "sensor": sensor, "sesion_sensor": sesion_sensor, "evaluador": evaluador, "estadio": estadio, "epoca": epoca, "etiqueta": etiqueta}
with conn_oltp:   # Transacción: confirma todo (COMMIT) o revierte todo (ROLLBACK)
    for nombre, df in TABLAS_OLTP.items():
        insertar(conn_oltp, nombre, df)

for nombre in TABLAS_OLTP:
    n = conn_oltp.execute(f"SELECT COUNT(*) FROM {nombre}").fetchone()[0]
    print(f"  {nombre:<14} {n:>8,} filas")
print(f"Tamaño de la base: {RUTA_OLTP.stat().st_size / 1024**2:.1f} MB")

  persona             100 filas
  sesion              128 filas
  dispositivo           2 filas
  sensor               10 filas
  sesion_sensor       965 filas
  evaluador             3 filas
  estadio               5 filas
  epoca           120,095 filas
  etiqueta        360,285 filas
Tamaño de la base: 19.2 MB


### 3.5 Validación

1. **Integridad referencial:** `PRAGMA foreign_key_check` no debe devolver filas huérfanas.
2. **Consistencia:** La base debe rechazar filas que violan las restricciones.
3. **Ida y vuelta:** Al reconstruir la tabla de épocas desde la base con `JOIN` y una pivot en SQL, el resultado debe ser idéntico a lo que se cargó. Lo mismo se verifica para la configuración de sensores: la tabla puente, vuelta a formato ancho, debe reproducir `channels.tsv`. Esto demuestra que la normalización no perdió información.

In [35]:
assert conn_oltp.execute("PRAGMA foreign_key_check").fetchall() == []
print("✓ Sin violaciones de integridad referencial")

pruebas = {
    "estadio inexistente (7)":      "UPDATE etiqueta SET estadio_id = 7 "
                                    "WHERE sesion_id = 'sub-1' AND nro_epoca = 0 AND evaluador_id = 1",
    "época de una sesión inexistente": "INSERT INTO epoca VALUES ('sub-999', 0, 0)",
    "sexo fuera del dominio":       "INSERT INTO persona VALUES (999, 'X')",
}
for caso, sql in pruebas.items():
    try:
        with conn_oltp:
            conn_oltp.execute(sql)
        print(f"✗ Se aceptó: {caso}")
    except sqlite3.IntegrityError as error:
        print(f"✓ Rechazado ({caso}): {error}")

SQL_RECONSTRUCCION = """
SELECT e.sesion_id AS participant_id, e.onset_s AS onset,
       MAX(CASE WHEN t.evaluador_id = 1 THEN t.estadio_id END) AS stage_hum,
       MAX(CASE WHEN t.evaluador_id = 2 THEN t.estadio_id END) AS stage_ai_psg,
       MAX(CASE WHEN t.evaluador_id = 3 THEN t.estadio_id END) AS stage_ai_hb
FROM epoca AS e
JOIN etiqueta AS t USING (sesion_id, nro_epoca)
GROUP BY e.sesion_id, e.nro_epoca
"""
reconstruida = pd.read_sql(SQL_RECONSTRUCCION, conn_oltp)
original = epocas[["participant_id", "onset"] + COLS_ESTADIO].astype({"participant_id": str})
clave = ["participant_id", "onset"]
pd.testing.assert_frame_equal(
    reconstruida.sort_values(clave).reset_index(drop=True).astype({c: "Int8" for c in COLS_ESTADIO}),
    original.sort_values(clave).reset_index(drop=True).astype({c: "Int8" for c in COLS_ESTADIO}),
    check_dtype=False)
print(f"✓ Ida y vuelta: las {len(reconstruida):,} épocas reconstruidas desde SQL coinciden con las cargadas")

sensores_sql = pd.read_sql("""
    SELECT ss.sesion_id AS REC, s.codigo
    FROM sesion_sensor AS ss JOIN sensor AS s USING (sensor_id)""", conn_oltp)
ancho = (pd.crosstab(sensores_sql["REC"], sensores_sql["codigo"])
         .reindex(index=canales["REC"], columns=COLS_SENSOR, fill_value=0))
assert (ancho.to_numpy() == canales[COLS_SENSOR].to_numpy()).all()
print("✓ Ida y vuelta: la tabla puente reproduce channels.tsv")

✓ Sin violaciones de integridad referencial
✓ Rechazado (estadio inexistente (7)): FOREIGN KEY constraint failed
✓ Rechazado (época de una sesión inexistente): FOREIGN KEY constraint failed
✓ Rechazado (sexo fuera del dominio): CHECK constraint failed: sexo IN ('F', 'M')
✓ Ida y vuelta: las 120,095 épocas reconstruidas desde SQL coinciden con las cargadas
✓ Ida y vuelta: la tabla puente reproduce channels.tsv


---

## 4. Almacén de datos (modelado dimensional)

La base transaccional anterior está optimizada para registrar datos sin redundancia. En cambio, el almacén está pensado para **analizar los datos**: organiza las medidas alrededor de las dimensiones desde las que se las quiere observar. Se diseña a partir de las siguientes preguntas que se quiere responder:

1. ¿Cuánto coincide cada IA con el consenso de expertos, en total y por estadio de sueño?
2. ¿Cómo varía esa concordancia según la edad, el sexo y el IMC de la persona?
3. ¿En qué momento de la noche falla más cada dispositivo?
4. ¿Cómo es la arquitectura del sueño de cada grupo de personas?

### 4.1 Diseño: esquema estrella (constelación)

El esquema (DDL) del almacén está en [`src/dw_esquema.sql`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/dw_esquema.sql).

![Esquema estrella del almacén de datos](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/img/estrella_dw.png?raw=true)

**Tablas de hechos.** Dos tablas de hechos comparten la dimensión `DIM_SESION` (constelación):

- `FACT_CONCORDANCIA`. Grano: sesión × IA × estadio del consenso × estadio de la IA × tercio de la noche. Su medida `n_epocas` es **aditiva**: es una matriz de confusión precalculada en forma de cubo. El % de acuerdo y el κ de Cohen **no son aditivos** (el promedio de κ por noche no es el κ del grupo), así que no se guardan agregados: se calculan al consultar, sumando conteos para cualquier combinación de filtros.
- `FACT_NOCHE`. Grano: una noche. Indicadores de arquitectura del sueño, calidad de señal y concordancia por noche.

**Dimensiones.**
- `DIM_SESION` está **desnormalizada a propósito**: reúne en una sola tabla los atributos de la persona (sexo), de la noche (edad, IMC, marcas de validez), los grupos derivados y la configuración de sensores aplanada en columnas sí/no. Los faltantes de edad e IMC quedan como un miembro **"Sin dato"**.
- `DIM_ESTADIO` cumple **dos roles** en `FACT_CONCORDANCIA`: estadio de referencia (consenso) y estadio predicho (IA). Incluye un miembro "Sin etiqueta válida" para las épocas con artefactos, que así también se pueden contar.
- `DIM_EVALUADOR` incluye los atributos del dispositivo. `DIM_TERCIO` ubica cada época en la noche.

Cada dimensión usa una **llave subrogada** (`*_key`), un entero propio del almacén e independiente de los identificadores de la fuente. Esto contrasta con la base transaccional, donde se normalizó hasta 3FN.

In [ ]:
RUTA_DW = DIR_DB / "boas_dw.db"
RUTA_DW.unlink(missing_ok=True)

# Sensores que varían entre noches (los presentes en todas no aportan información)
SENSORES_VARIABLES = [c for c in COLS_SENSOR if canales[c].nunique() > 1]
cols_sensores = ",\n    ".join(f"tiene_{c.lower()} INTEGER NOT NULL" for c in SENSORES_VARIABLES)

DDL_DW = leer_sql("dw_esquema.sql", columnas_sensores=cols_sensores)
conn_dw = sqlite3.connect(RUTA_DW)
conn_dw.execute("PRAGMA foreign_keys = ON")
conn_dw.executescript(DDL_DW)
conn_dw.execute(f"ATTACH DATABASE '{RUTA_OLTP}' AS oltp")   # La base transaccional es la fuente
print("Tablas del almacén:", [t for (t,) in conn_dw.execute(
    "SELECT name FROM main.sqlite_master WHERE type = 'table' ORDER BY rowid")])

### 4.2 ETL desde la base transaccional (SQL)

Las sentencias de carga están en [`src/dw_etl.sql`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/dw_etl.sql), agrupadas en cuatro bloques comentados dentro de una única transacción (`BEGIN … COMMIT`).

El almacén se carga directamente desde la base transaccional (`ATTACH`) con sentencias `INSERT … SELECT`, de modo que todas las transformaciones quedan expresadas en SQL:

- **Dimensiones:** El grupo etario y la categoría de IMC se derivan con `CASE` (con "Sin dato" para los `NULL`). Los sensores se aplanan con `MAX(CASE …)` a partir de la tabla puente.
- **`FACT_CONCORDANCIA`:** Se enfrenta cada etiqueta de IA con la del consenso de la misma época (*self-join* de `etiqueta`), el tercio se calcula con una función de ventana (`COUNT(*) OVER (PARTITION BY sesion_id)`) y se agrega con `GROUP BY`. Los `NULL` se emparejan con el miembro "Sin etiqueta válida" usando el operador `IS`.
- **`FACT_NOCHE`:** La arquitectura del sueño se calcula sobre las etiquetas del consenso. El κ por noche sale de la propia matriz de confusión: κ = (pₒ − pₑ) / (1 − pₑ), donde pₒ es la proporción de acuerdo y pₑ el acuerdo esperado por azar según los marginales.

In [ ]:
columnas = ", ".join(f"tiene_{c.lower()}" for c in SENSORES_VARIABLES)
flags = ",\n       ".join(f"MAX(CASE WHEN se.codigo = '{c}' THEN 1 ELSE 0 END)" for c in SENSORES_VARIABLES)
conn_dw.executescript(leer_sql("dw_etl.sql", columnas_sensores=columnas, flags_sensores=flags))

for (t,) in conn_dw.execute("SELECT name FROM main.sqlite_master WHERE type = 'table' ORDER BY rowid"):
    print(f"  {t:<18} {conn_dw.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>7,} filas")

### 4.3 Validación

Se emplea la validación de lo generado anteriormente mediante:

1. **Integridad referencial** de las llaves subrogadas.
2. **Conservación de las épocas:** El cubo debe sumar exactamente las 120.095 épocas por evaluador, y la distribución por tercio debe coincidir con la calculada en la Transformación.
3. **Dos implementaciones independientes:** Los grupos etarios derivados en SQL deben coincidir con los de pandas, y los indicadores de `FACT_NOCHE` (calculados en SQL, incluido el κ) deben coincidir con los de la Transformación (pandas + scikit-learn).

In [38]:
assert conn_dw.execute("PRAGMA foreign_key_check").fetchall() == []

por_evaluador = pd.read_sql("""SELECT de.descripcion, SUM(f.n_epocas) AS epocas
    FROM fact_concordancia f JOIN dim_evaluador de USING (evaluador_key) GROUP BY 1""", conn_dw)
assert (por_evaluador["epocas"] == len(epocas)).all()

tercio_sql = pd.read_sql("""SELECT ds.sesion_id, f.tercio_key AS tercio, SUM(f.n_epocas) AS n
    FROM fact_concordancia f JOIN dim_sesion ds USING (sesion_key)
    WHERE f.evaluador_key = 1 GROUP BY 1, 2""", conn_dw).set_index(["sesion_id", "tercio"])["n"]
tercio_pd = epocas.groupby(["participant_id", "tercio"], observed=True).size()
assert tercio_sql.sort_index().to_numpy().tolist() == tercio_pd.rename_axis(["sesion_id", "tercio"]).sort_index().to_numpy().tolist()
print("✓ Integridad referencial, conservación de épocas y tercios")

dim = pd.read_sql("SELECT sesion_id, grupo_etario, categoria_imc FROM dim_sesion", conn_dw).set_index("sesion_id")
ref = noches.set_index("participant_id")[["grupo_etario", "categoria_imc"]].apply(
    lambda s: s.astype("string").fillna("Sin dato"))
assert (dim.loc[ref.index].to_numpy() == ref.to_numpy()).all()

fact_noche = pd.read_sql("SELECT ds.sesion_id, fn.* FROM fact_noche fn JOIN dim_sesion ds USING (sesion_key)",
                         conn_dw).set_index("sesion_id")
COMUNES = ["horas_registro", "tst_min", "eficiencia", "latencia_sueno_min", "latencia_rem_min",
           "waso_min", "pct_artefacto_hb", "kappa_psg", "kappa_hb"]
esperado = indicadores[COMUNES].astype(float).set_axis(indicadores.index.astype(str))
pd.testing.assert_frame_equal(fact_noche.loc[esperado.index, COMUNES], esperado,
                              check_exact=False, atol=1e-9, check_names=False)
print("✓ Grupos y los 9 indicadores por noche coinciden entre SQL y pandas/scikit-learn")

✓ Integridad referencial, conservación de épocas y tercios
✓ Grupos y los 9 indicadores por noche coinciden entre SQL y pandas/scikit-learn


### 4.4 Consultas analíticas (OLAP)

La consulta del cubo está en [`src/dw_cubo.sql`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/dw_cubo.sql) y la función `concordancia()` en [`src/metricas.py`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/metricas.py).

Todas las consultas parten del mismo cubo: los hechos unidos a sus dimensiones, con las épocas válidas en ambas etiquetas y solo las noches válidas para cada dispositivo (criterio de 4 h de la Transformación). Como las medidas de concordancia no son aditivas, se calculan después de agregar los conteos, para cada corte (*slice*) del cubo.

In [ ]:
datos_cubo = pd.read_sql(leer_sql("dw_cubo.sql"), conn_dw)
cubo = datos_cubo[datos_cubo["noche_valida"] == 1]          # Solo noches válidas para cada dispositivo

print("1) Concordancia global por dispositivo")
print(cubo.groupby("dispositivo").apply(concordancia, include_groups=False)
      .astype({"epocas": "int64"}).round(3).to_string())

ORDEN_ESTADIOS = ["W", "N1", "N2", "N3", "REM"]
sensibilidad = (cubo.assign(acierto=lambda d: d["n"].where(d["referencia"] == d["prediccion"], 0))
                .groupby(["dispositivo", "referencia"])[["acierto", "n"]].sum()
                .eval("acierto / n").unstack("referencia")[ORDEN_ESTADIOS])
print("\n2) Sensibilidad por estadio (proporción de épocas de cada estadio que la IA reconoce)")
print(sensibilidad.round(3).to_string())

print("\n3) κ por grupo etario y dispositivo")
print(cubo.groupby(["grupo_etario", "dispositivo"]).apply(concordancia, include_groups=False)["kappa"]
      .unstack().round(3).to_string())

---

## 5. Visualización: tablero interactivo

El código del tablero está en [`src/tablero.py`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/src/tablero.py): los tres gráficos, las tarjetas de κ, los filtros y el callback que recalcula todo al cambiar un filtro. La notebook solo le pasa el cubo leído del almacén.

**Idea:** *La vincha wearable estadifica el sueño casi tan bien como el PSG clínico, pero pierde precisión en el sueño profundo (fase N3).*

El tablero se alimenta únicamente del almacén de datos (una consulta al cubo de concordancia) y recalcula todas las medidas para la combinación de filtros elegida. Decisiones de diseño, siguiendo los principios del curso:

- **El título es la conclusión**, no una descripción de los ejes.
- **Números grandes** para el resultado principal: el κ de cada dispositivo.
- **Atributos preatentivos:** en el gráfico de sensibilidad solo N3 se muestra con opacidad plena y con su diferencia anotada; los demás estadios quedan atenuados como contexto.
- **Un color fijo por dispositivo** (azul = PSG, naranja = vincha) en todos los gráficos, de una paleta validada para daltonismo, siempre acompañado de leyenda y etiquetas.
- **Poca tinta:** fondo blanco, sin grillas innecesarias, etiquetas directas solo donde aportan.

**Filtros:** sexo, grupo etario, categoría de IMC, tercio de la noche y la opción de incluir las noches con menos de 4 h de señal válida (excluidas por defecto, según la decisión de la Transformación). 

La matriz de confusión se puede ver para cada dispositivo. La noche sin edad informada (`sub-16`) y la sin IMC (`sub-29`) quedan fuera cuando se filtra por esas variables.

In [ ]:
app = crear_app(datos_cubo)    # Cubo leído del almacén en 4.4 (con todas las noches; el filtro de calidad es interactivo)
app.run(jupyter_mode="inline", jupyter_height=1250, port=8050)

**Vista estática del tablero** (con todos los filtros activos y solo las noches válidas). _Se incluye porque los visores que no ejecutan el cuaderno, como GitHub, no muestran la aplicación interactiva._

![Vista estática del tablero](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/img/tablero.png?raw=true)

---